In [2]:
import os
from datetime import datetime, timezone

import requests

In [3]:
API_KEY = 'aDiFX6Um0eZEEDRQBN4hrHGzIto9xnoDjXebmr87'
print(f'API key cargada: {API_KEY[:4]}{'*' * (len(API_KEY) - 4)}')

API key cargada: aDiF************************************


In [5]:
NOMBRE = input('Nombre completo: ').strip()
CARNET = input('Carnet: ').strip()

if not NOMBRE or not CARNET:
    raise ValueError('Debes ingresar nombre y carnet para continuar.')

print(f'Estudiante: {NOMBRE}')
print(f'Carnet    : {CARNET}')

Nombre completo: Sahara Alessandra Farfán Torres
Carnet: 202308315
Estudiante: Sahara Alessandra Farfán Torres
Carnet    : 202308315


In [4]:
URL_BASE = 'https://api.nasa.gov/DONKI/GST'
START_DATE = '2024-05-01'
END_DATE = '2024-05-31'

def construir_url(base, start_date, end_date, api_key):
    return f'{base}?startDate={start_date}&endDate={end_date}&api_key={api_key}'

url = construir_url(URL_BASE, START_DATE, END_DATE, API_KEY)
respuesta = requests.get(url, timeout=15)

if not respuesta.ok:
    print(f'Error HTTP {respuesta.status_code}: {respuesta.text}')
    raise SystemExit('No se pudo obtener la respuesta. Intenta de nuevo.')

tormentas = respuesta.json()
print(f'Solicitud exitosa: {len(tormentas)} tormentas recibidas')

Solicitud exitosa: 5 tormentas recibidas


In [6]:
print(f'Total de tormentas en el periodo: {len(tormentas)}')
print('-' * 80)
for i, tormenta in enumerate(tormentas, start=1):
    print(f'{i:>2}. {tormenta["gstID"]} | {tormenta["startTime"]}')

Total de tormentas en el periodo: 5
--------------------------------------------------------------------------------
 1. 2024-05-02T15:00:00-GST-001 | 2024-05-02T15:00Z
 2. 2024-05-10T15:00:00-GST-001 | 2024-05-10T15:00Z
 3. 2024-05-12T21:00:00-GST-001 | 2024-05-12T21:00Z
 4. 2024-05-16T06:00:00-GST-001 | 2024-05-16T06:00Z
 5. 2024-05-17T18:00:00-GST-001 | 2024-05-17T18:00Z


In [7]:
def extraer_kps(tormenta):
    return [m['kpIndex'] for m in tormenta['allKpIndex']]

print('Kp maximo por tormenta')
print('-' * 60)
for tormenta in tormentas:
    kp_max = max(extraer_kps(tormenta))
    print(f"{tormenta['startTime']} -> Kp max = {kp_max:.2f}")

tormenta_top = None
kp_global_max = float('-inf')

for tormenta in tormentas:
    kp_max = max(extraer_kps(tormenta))
    if kp_max > kp_global_max:
        kp_global_max = kp_max
        tormenta_top = tormenta

print('\nTormenta mas intensa del periodo')
print('-' * 60)
print(f"ID     : {tormenta_top['gstID']}")
print(f"Inicio : {tormenta_top['startTime']}")
print(f'Kp max : {kp_global_max:.2f}')

Kp maximo por tormenta
------------------------------------------------------------
2024-05-02T15:00Z -> Kp max = 6.67
2024-05-10T15:00Z -> Kp max = 9.00
2024-05-12T21:00Z -> Kp max = 6.33
2024-05-16T06:00Z -> Kp max = 6.00
2024-05-17T18:00Z -> Kp max = 6.00

Tormenta mas intensa del periodo
------------------------------------------------------------
ID     : 2024-05-10T15:00:00-GST-001
Inicio : 2024-05-10T15:00Z
Kp max : 9.00


In [8]:
def clasificar_kp(kp):
    kp_entero = int(kp)
    if kp_entero < 4:
        return 'Quieto'
    if kp_entero == 4:
        return 'Activo'
    if kp_entero == 5:
        return 'G1 - Menor'
    if kp_entero == 6:
        return 'G2 - Moderada'
    if kp_entero == 7:
        return 'G3 - Fuerte'
    if kp_entero == 8:
        return 'G4 - Severa'
    return 'G5 - Extrema'

print('Pruebas clasificar_kp')
for kp_prueba in [2.0, 4.0, 5.33, 6.67, 7.0, 8.67, 9.0]:
    print(f'kp={kp_prueba:.2f} -> {clasificar_kp(kp_prueba)}')

Pruebas clasificar_kp
kp=2.00 -> Quieto
kp=4.00 -> Activo
kp=5.33 -> G1 - Menor
kp=6.67 -> G2 - Moderada
kp=7.00 -> G3 - Fuerte
kp=8.67 -> G4 - Severa
kp=9.00 -> G5 - Extrema


In [9]:

print(f"{'Fecha inicio':<22} {'Kp max':>7}  Categoria")
print('-' * 50)
for tormenta in tormentas:
    kp_max = max(extraer_kps(tormenta))
    categoria = clasificar_kp(kp_max)
    print(f"{tormenta['startTime']:<22} {kp_max:>7.2f}  {categoria}")

Fecha inicio            Kp max  Categoria
--------------------------------------------------
2024-05-02T15:00Z         6.67  G2 - Moderada
2024-05-10T15:00Z         9.00  G5 - Extrema
2024-05-12T21:00Z         6.33  G2 - Moderada
2024-05-16T06:00Z         6.00  G2 - Moderada
2024-05-17T18:00Z         6.00  G2 - Moderada


In [10]:
def analizar_tormenta(tormenta):
    valores_kp = extraer_kps(tormenta)
    kp_max = max(valores_kp)
    kp_min = min(valores_kp)
    kp_promedio = round(sum(valores_kp) / len(valores_kp), 2)

    return {
        'id': tormenta['gstID'],
        'inicio': tormenta['startTime'],
        'kp_max': kp_max,
        'kp_min': kp_min,
        'kp_promedio': kp_promedio,
        'num_mediciones': len(valores_kp),
        'categoria': clasificar_kp(kp_max),
        'eventos_vinculados': len(tormenta['linkedEvents'] or []),
    }

print('Resumen por tormenta')
print('=' * 60)
for tormenta in tormentas:
    r = analizar_tormenta(tormenta)
    print(f"ID                : {r['id']}")
    print(f"Inicio            : {r['inicio']}")
    print(f"Kp max            : {r['kp_max']:.2f}")
    print(f"Kp min            : {r['kp_min']:.2f}")
    print(f"Kp promedio       : {r['kp_promedio']:.2f}")
    print(f"Num mediciones    : {r['num_mediciones']}")
    print(f"Categoria         : {r['categoria']}")
    print(f"Eventos vinculados: {r['eventos_vinculados']}")
    print('-' * 60)

Resumen por tormenta
ID                : 2024-05-02T15:00:00-GST-001
Inicio            : 2024-05-02T15:00Z
Kp max            : 6.67
Kp min            : 6.67
Kp promedio       : 6.67
Num mediciones    : 2
Categoria         : G2 - Moderada
Eventos vinculados: 2
------------------------------------------------------------
ID                : 2024-05-10T15:00:00-GST-001
Inicio            : 2024-05-10T15:00Z
Kp max            : 9.00
Kp min            : 6.67
Kp promedio       : 8.13
Num mediciones    : 13
Categoria         : G5 - Extrema
Eventos vinculados: 6
------------------------------------------------------------
ID                : 2024-05-12T21:00:00-GST-001
Inicio            : 2024-05-12T21:00Z
Kp max            : 6.33
Kp min            : 5.67
Kp promedio       : 6.00
Num mediciones    : 3
Categoria         : G2 - Moderada
Eventos vinculados: 2
------------------------------------------------------------
ID                : 2024-05-16T06:00:00-GST-001
Inicio            : 2024-05-16T

In [11]:
print('Consultando posicion de la ISS...')
try:
    r_iss = requests.get('http://api.open-notify.org/iss-now.json', timeout=8)
    r_iss.raise_for_status()
    iss_data = r_iss.json()

    iss_lat = float(iss_data['iss_position']['latitude'])
    iss_lon = float(iss_data['iss_position']['longitude'])
    iss_hora = datetime.fromtimestamp(iss_data['timestamp'], tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
    iss_ok = True
except Exception as e:
    iss_lat, iss_lon, iss_hora = 0.0, 0.0, 'no disponible'
    iss_ok = False
    print(f'Advertencia: no se pudo obtener posicion ISS ({e})')

ts_entrega = datetime.now(tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

print('\n' + '=' * 55)
print('           SELLO DE ENTREGA - PARCIAL 2')
print('=' * 55)
print(f'Estudiante   : {NOMBRE}')
print(f'Carnet       : {CARNET}')
print(f'Fecha/Hora   : {ts_entrega}')
print(f'ISS Latitud  : {iss_lat:+.4f} deg')
print(f'ISS Longitud : {iss_lon:+.4f} deg')
print(f'ISS Hora UTC : {iss_hora}')
print('=' * 55)

if not iss_ok:
    print('Advertencia: sello generado sin datos de ISS (sin conexion).')

Consultando posicion de la ISS...

           SELLO DE ENTREGA - PARCIAL 2
Estudiante   : Sahara Alessandra Farfán Torres
Carnet       : 202308315
Fecha/Hora   : 2026-04-24 05:56:10 UTC
ISS Latitud  : -51.5809 deg
ISS Longitud : +172.9979 deg
ISS Hora UTC : 2026-04-24 05:56:10 UTC
